In [6]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

In [6]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: en-zh.en-filtered-salient.en.subword.train
        path_tgt: en-zh.zh-filtered.zh.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: en-zh.en-filtered-salient.en.subword.dev
        path_tgt: en-zh.zh-filtered.zh.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 10000
tgt_vocab_size: 10000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 512
src_seq_length: 512

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.fren

# Stop training if it does not imporve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 2000

# To save space, limit checkpoints to last n
# keep_checkpoint: 3

seed: 3435

# Default: 100000 - Train the model to max n steps 
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 10000

# Default: 10000 - Run validation after n steps
valid_steps: 2000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 4000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"
weight_decay: 0.0001

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)

In [7]:
# Find the number of CPUs/cores on the machine
!nproc --all

60


In [9]:
# Build Vocabulary

# -config: path to your config.yaml file
# -n_sample: use -1 to build vocabulary on all the segment in the training dataset
# -num_threads: change it to match the number of CPUs to run it faster

!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 60


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_build_vocab", line 5, in <module>
    from onmt.bin.build_vocab import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "

In [10]:
# Check if the GPU is active
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-a2b8d06e-289e-5f2e-9f0d-3f5bd1eb21cd)


In [11]:
# Check if the GPU is visable to PyTorch
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

gpu_memory = torch.cuda.mem_get_info(0)
print("Free GPU memory:", gpu_memory[0]/1024**2, "out of:", gpu_memory[1]/1024**2)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/venv/main/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/venv/main/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/venv/main/lib/python3.10/site-packages/ipykernel/kernelapp.p

True
NVIDIA A100-SXM4-40GB
Free GPU memory: 40021.375 out of: 40444.375


In [12]:
# Train the NMT model
!onmt_train -config config.yaml


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_train", line 5, in <module>
    from onmt.bin.train import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "/venv/main/l

## Translate

In [25]:
# Translate the "subworded" source file of the test dataset
# Change the model name, if needed.
# gpu
!onmt_translate -model models/model.fren_step_10000.pt -src en-zh.en-filtered-salient.en.subword.test -output zh.translated -gpu 0 -min_length 1



[2025-04-01 01:03:42,006 INFO] Loading checkpoint from models/model.fren_step_10000.pt
[2025-04-01 01:03:43,843 INFO] Loading data into the model
[2025-04-01 01:03:55,387 INFO] PRED SCORE: -0.6327, PRED PPL: 1.88 NB SENTENCES: 2000
Time w/o python interpreter load/terminate:  13.387151002883911


In [14]:
%pip install "numpy<2"


Note: you may need to restart the kernel to use updated packages.


In [26]:
# Check the first 5 lines of the translation file
!head -n 30 zh.translated

▁在我 还没有 意识到 之前 , ▁她 已经 跨越 了 停车场 和 车 间 , ▁在我 身 后 的人 , ▁ 带着 这样的 “ 我要 过来 ” , 意大利 式的 手 势 跟随 了 。
▁如果我 把 乌干达 分割 开 , 乌干达 境 内 也 有很大的 差别 。
▁我 看着 这张照片 , ▁他 似乎 对 那个 按钮 上 有什么 特别 感兴趣 , ▁但是 看起来 他 对 过 马路 ▁ 没 那么 感兴趣 。
▁我不知道 , ▁ 这个项目 有很多 可能性 , ▁我 鼓励 你们 每天 记录 一 小 段 ▁ 生活中 的小 片段 , ▁因此 你 永远 忘 不了 那个 一天 , ▁ 生活 就 活 下去 了 。
▁它 来自 一种 名为 P a ch y ma 的 <unk> 疮 。
▁ 回到 希 思 罗 机场 后 , 我 朋友 在那里 , ▁我 的 哥哥 在那里 , 我的 孙 子 在那里 —— ▁ 跟 J a ck 在一起 —— ▁就是 这个 。 我 回到 我的 妈妈 身边 。
▁这不是 一个 修 辞 的问题 。
▁ 好消息 是 , 这些 领导人 大部分 都 已经 搬 进来 了 , ▁他们 被 三 代 取代 。
▁ 前 几天 , 我 和 他 聊 聊 了 聊 , ▁当时 他 在 弗 罗 里 达 大学 ▁ 读 了他的 公共 卫生 博士 , ▁他 很 自豪 地 告诉我 , ▁他 是如何 从 美国 公共 机构 ▁ 筹 到 足够 的 资金 ▁来 在 自己的 村庄 建立 诊所 。 ▁我想 带 你 回到 汉 尼 。
▁我 只是想 很快 地 展示 给你们看 。
▁克里斯 · 安德森 : 但是 我所 理解 的 弦 理论 , ▁能 解释 更 小的 弦 弦 弦 理论 的 电子 -- ▁我知道 你们 不喜欢 弦 理论 -- 在 震动 中 振 动 。
▁ 另一 条 我喜欢 的 菜 式 : ▁ 左 宗 棠 鸡 —— ▁ 顺便 说 一下 , 在美国 纳 瓦 西 学院 ▁ 叫做 阿 迪 米 · 塔 斯 科 。
▁ 按照 我们的 标准 , ▁有些人 会把 这些人 看 做 是 原始 的 。
▁当 这些 我们 习惯 的方式 ▁ 看到 我们的 同 胞 们 ▁被 尊 心 、 恐惧 和 怀疑 所 吸引 。
▁于是我 说 ,“ 哦 , 这是 额外 的 成分 , ▁你知道 , 你 也需要 爸爸 和 青蛙 。” ▁她说 ,“ 哦 , 

In [27]:
# If needed install/update sentencepiece
!pip3 install --upgrade -q sentencepiece

# Desubword the translation file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model zh.translated

Done desubwording! Output: zh.translated.desubword


In [32]:
# Desubword the target file (reference) of the test dataset
# Note: You might as well have split files *before* subwording during dataset preperation, 
# but sometimes datasets have tokeniztion issues, so this way you are sure the file is really untokenized.
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model en-zh.en-filtered-salient.en.subword.test

# Desubword the test file
!python3 ./MT-Preparation/subwording/3-desubword.py ./source.model en-zh.en-filtered-salient.en.subword.test

Done desubwording! Output: en-zh.en-filtered-salient.en.subword.test.desubword
Done desubwording! Output: en-zh.en-filtered-salient.en.subword.test.desubword


In [35]:
# Check the first 5 lines of the desubworded translation file
!head -n 30 zh.translated.desubword

print("---------------")
!python3 ./MT-Preparation/subwording/3-desubword.py ./source.model en-zh.zh-filtered.zh.subword.test
# Check the first 5 lines of the desubworded reference
!head -n 30 en-zh.zh-filtered.zh.subword.test.desubword

在我还没有意识到之前, 她已经跨越了停车场和车间, 在我身后的人, 带着这样的“我要过来”,意大利式的手势跟随了。
如果我把乌干达分割开,乌干达境内也有很大的差别。
我看着这张照片, 他似乎对那个按钮上有什么特别感兴趣, 但是看起来他对过马路 没那么感兴趣。
我不知道, 这个项目有很多可能性, 我鼓励你们每天记录一小段 生活中的小片段, 因此你永远忘不了那个一天, 生活就活下去了。
它来自一种名为Pachyma的 ⁇ 疮。
回到希思罗机场后,我朋友在那里, 我的哥哥在那里,我的孙子在那里—— 跟Jack在一起—— 就是这个。我回到我的妈妈身边。
这不是一个修辞的问题。
好消息是,这些领导人大部分都已经搬进来了, 他们被三代取代。
前几天,我和他聊聊了聊, 当时他在弗罗里达大学 读了他的公共卫生博士, 他很自豪地告诉我, 他是如何从美国公共机构 筹到足够的资金 来在自己的村庄建立诊所。 我想带你回到汉尼。
我只是想很快地展示给你们看。
克里斯·安德森:但是我所理解的弦理论, 能解释更小的弦弦弦理论的电子-- 我知道你们不喜欢弦理论--在震动中振动。
另一条我喜欢的菜式: 左宗棠鸡—— 顺便说一下,在美国纳瓦西学院 叫做阿迪米·塔斯科。
按照我们的标准, 有些人会把这些人看做是原始的。
当这些我们习惯的方式 看到我们的同胞们 被尊心、恐惧和怀疑所吸引。
于是我说,“哦,这是额外的成分, 你知道,你也需要爸爸和青蛙。” 她说,“哦,对人类来说也一样?”
这是一位社会科学家的捐赠, 他们说他们不知道该用什么, 对不起。
你在网上看,你会看到一些地方, 比如科德美术和事件, 比如多约少女会之类的网站, 比如“女孩子”,或者“黑人女孩子”。
我可以用她一起出笑。
直到那天,我站在我父亲面前, 试图用一把老枪射杀纳粹。
于是我找了警察来确认Abed 仍然住在他同一个镇里 现在我用一个装满了黄色的玫瑰花 坐在后座上 突然觉得花很荒唐
没有什么时间可说,但是其中包括 树丛之类的动物园, 农田的耕作方式, 类似的山峰的土地。
我们都必须感到重要, 特别,特殊。
所以对于我,作为一个公共卫生教授, 一点也不奇怪,所有的这些国家都发展得那么快。
所以我们一直在问的一个问题是, 世界有多少被用来种植粮食, 目的是什么, 以及我们如何改变未来, 这对什么意义?
一些与压迫的政府作战。
当她的情

In [37]:
def clean_sentencepiece_output(line):
    return line.replace('▁', ' ').strip()

# Example
with open("en-zh.zh-filtered.zh.subword.test.desubword", "r") as infile:
    lines = infile.readlines()

cleaned = [clean_sentencepiece_output(line) for line in lines]

with open("en-zh.zh-filtered.zh.subword.test.cleaned", "w") as outfile:
    outfile.write("\n".join(cleaned))


## Evaluation

In [21]:
# Download the BLEU script
!wget https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py

--2025-04-01 01:01:43--  https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
200 OKequest sent, awaiting response... 
Length: 957 [text/plain]
Saving to: ‘compute-bleu.py.1’

compute-bleu.py.1   100%[===================>]     957  --.-KB/s    in 0s      

2025-04-01 01:01:44 (36.4 MB/s) - ‘compute-bleu.py.1’ saved [957/957]



In [22]:
# Install sacrebleu
!pip3 install sacrebleu

In [38]:
# Evaluate the translation (without subwording)
!python3 compute-bleu.py en-zh.zh-filtered.zh.subword.test.cleaned zh.translated.desubword

Reference 1st sentence: 我还没回过神儿,她已轻松穿行于停车场的汽车中, 身后的人们看我的眼神, 充满了惊羡。哇啊——哇啊——
MTed 1st sentence: 在我还没有意识到之前, 她已经跨越了停车场和车间, 在我身后的人, 带着这样的“我要过来”,意大利式的手势跟随了。
BLEU:  2.662938315506583
